# 🌾 Crop Yield Predictor
### Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

print('Libraries imported!')

### Step 2 — Create Dataset

In [ ]:
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'temperature_c':  np.clip(np.random.normal(28, 7, n), 10, 45),
    'rainfall_mm':    np.clip(np.random.exponential(120, n), 20, 500),
    'humidity_pct':   np.clip(np.random.normal(65, 15, n), 20, 100),
    'nitrogen':       np.clip(np.random.normal(80, 25, n), 10, 150),
    'phosphorus':     np.clip(np.random.normal(50, 15, n), 5, 100),
    'potassium':      np.clip(np.random.normal(60, 20, n), 5, 120),
    'ph':             np.clip(np.random.normal(6.5, 0.8, n), 4.5, 8.5),
})

# Target: yield category
yield_score = (
    0.003 * df['rainfall_mm']
    + 0.005 * df['nitrogen']
    - 0.002 * abs(df['temperature_c'] - 28)
    + np.random.normal(0, 0.3, n)
)
df['yield_category'] = pd.cut(yield_score, bins=3, labels=['Low', 'Medium', 'High'])

print('Dataset created!')
print(df.head())
print('\nYield distribution:')
print(df['yield_category'].value_counts())

### Step 3 — Encode Target Column

In [ ]:
le = LabelEncoder()
df['yield_category'] = le.fit_transform(df['yield_category'].astype(str))

print('Classes:', le.classes_)  # High=0, Low=1, Medium=2
print(df.head())

### Step 4 — Split Data

In [ ]:
X = df.drop('yield_category', axis=1)
y = df['yield_category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train size:', X_train.shape)
print('Test size :', X_test.shape)

### Step 5 — Train Decision Tree Model

In [ ]:
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

print('Model trained!')

### Step 6 — Evaluate Model

In [ ]:
y_pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

### Step 7 — Export Model as .pkl

In [ ]:
with open('crop_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('crop_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('Model saved as crop_model.pkl')
print('Encoder saved as crop_encoder.pkl')

### Step 8 — Test Loading the Model

In [ ]:
with open('crop_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('crop_encoder.pkl', 'rb') as f:
    loaded_le = pickle.load(f)

# [temperature, rainfall, humidity, nitrogen, phosphorus, potassium, ph]
sample = [[28, 150, 70, 90, 55, 65, 6.5]]
pred = loaded_model.predict(sample)
label = loaded_le.inverse_transform(pred)

print('Predicted yield category:', label[0])
print('Model loading works perfectly!')